# Topic 18: Model Evaluation

**Topics:** Accuracy · Precision · Recall · F1 · Confusion Matrix · Regression metrics · Overfitting vs Underfitting


In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
np.random.seed(42)

X = pd.DataFrame({"A":np.random.randn(400),"B":np.random.randn(400),"C":np.random.randn(400),"D":np.random.randn(400)})
y = (X["A"] + X["B"] > 0).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
model = LogisticRegression(random_state=42).fit(X_tr_s, y_tr)
y_pred = model.predict(X_te_s)
print("Setup complete.")

## 1. The Four Classification Metrics

| Metric | Formula | Good when |
|---|---|---|
| **Accuracy** | Correct / Total | Classes are balanced |
| **Precision** | TP / (TP+FP) | False positives are costly |
| **Recall** | TP / (TP+FN) | False negatives are costly |
| **F1-Score** | 2×P×R/(P+R) | Need balance of precision & recall |


In [ ]:
print(f"Accuracy  : {accuracy_score(y_te, y_pred):.2%}")
print(f"Precision : {precision_score(y_te, y_pred):.2%}")
print(f"Recall    : {recall_score(y_te, y_pred):.2%}")
print(f"F1-Score  : {f1_score(y_te, y_pred):.2%}")

## 2. Confusion Matrix


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives : {tn}")
print(f"False Positives: {fp}  (predicted Yes but actually No)")
print(f"False Negatives: {fn}  (predicted No  but actually Yes)")
print(f"True Positives : {tp}")

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Pred 0","Pred 1"], yticklabels=["Actual 0","Actual 1"])
ax.set_title("Confusion Matrix")
plt.tight_layout(); plt.show()

## 3. Regression Metrics

| Metric | Measures |
|---|---|
| **MAE** | Average absolute prediction error (same unit as target) |
| **RMSE** | Like MAE but penalises large errors more |
| **R²** | Proportion of variance explained (1 = perfect) |


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

exp = np.random.randint(0,30,200).reshape(-1,1)
sal = 28000 + exp.ravel()*2400 + np.random.normal(0,5000,200)
e_tr,e_te,s_tr,s_te = train_test_split(exp,sal,test_size=0.2,random_state=42)
reg = LinearRegression().fit(e_tr, s_tr)
pred = reg.predict(e_te)

print(f"MAE : ${mean_absolute_error(s_te,pred):,.0f}")
print(f"RMSE: ${mean_squared_error(s_te,pred)**0.5:,.0f}")
print(f"R²  : {r2_score(s_te,pred):.3f}")

## 4. Overfitting vs Underfitting

| | Underfitting | Good Fit | Overfitting |
|---|---|---|---|
| Train accuracy | Low | High | Very High |
| Test accuracy | Low | High | Low |
| Gap | Small | Small | Large |


In [ ]:
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr_s, y_tr)
    train_acc.append(accuracy_score(y_tr, dt.predict(X_tr_s)))
    test_acc.append(accuracy_score(y_te, dt.predict(X_te_s)))

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(depths, train_acc, "o-", label="Train", color="blue")
ax.plot(depths, test_acc,  "s-", label="Test",  color="red")
ax.axvline(3, linestyle="--", color="green", alpha=0.7, label="Sweet spot")
ax.set_title("Overfitting vs Underfitting")
ax.set_xlabel("Tree Depth"); ax.set_ylabel("Accuracy")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Cross-Validation

More reliable than a single train/test split — trains and tests **k** times.


In [ ]:
from sklearn.model_selection import cross_val_score

for name, mdl in [("Logistic Regression", LogisticRegression(random_state=42)),
                   ("Decision Tree d=3",  DecisionTreeClassifier(max_depth=3, random_state=42))]:
    cv = cross_val_score(mdl, sc.fit_transform(X), y, cv=5, scoring="accuracy")
    print(f"{name:<25} mean={cv.mean():.2%}  std={cv.std():.3f}")

## ✅ Summary

- Use **accuracy** only when classes are balanced.
- Use **precision** when false positives are costly.
- Use **recall** when false negatives are costly.
- **F1** balances precision and recall.
- Always check training **and** test performance — a big gap means overfitting.
- Use **cross-validation** for a more reliable estimate.
